In [ ]:
from __future__ import annotations

import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:  # pragma: no cover - optional dependency import guard
    torch = None


OUTER_FOLDS = (1, 2, 3, 4, 5)
MODEL_FAMILY = "tabpfn"
MODEL_ID = "tabpfn_v26"
RUN_ROLE = "default"
TARGET_COL = "Target_Log"
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 2

build_prediction_frame = None
compute_regression_metrics = None
ensure_family_result_dir = None
load_full_feature_processed = None
save_model_artifact = None
summarize_fold_metrics = None
TabPFNRegressor = None
ModelVersion = None


In [ ]:
def resolve_current_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError(
            "cannot resolve current code directory without __file__; "
            "current working directory must be the Step5 1_Code directory"
        )
    return code_dir


In [ ]:
def initialize_runtime() -> Path:
    global TARGET_COL
    global build_prediction_frame
    global compute_regression_metrics
    global ensure_family_result_dir
    global load_full_feature_processed
    global save_model_artifact
    global summarize_fold_metrics
    global TabPFNRegressor
    global ModelVersion

    current_code_dir = resolve_current_code_dir()
    if str(current_code_dir) not in sys.path:
        sys.path.insert(0, str(current_code_dir))

    try:
        from tabpfn import TabPFNRegressor as runtime_tabpfn_regressor
        from tabpfn.constants import ModelVersion as runtime_model_version
    except Exception as exc:  # pragma: no cover - runtime fallback for environments without tabpfn
        class _MissingModelVersion:
            V2_6 = "V2_6"

        class _MissingTabPFNRegressor:
            @staticmethod
            def create_default_for_version(*args, **kwargs):
                del args, kwargs
                raise RuntimeError("tabpfn runtime is unavailable") from exc

        runtime_tabpfn_regressor = _MissingTabPFNRegressor
        runtime_model_version = _MissingModelVersion

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        build_prediction_frame as shared_build_prediction_frame,
        compute_regression_metrics as shared_compute_regression_metrics,
        ensure_family_result_dir as shared_ensure_family_result_dir,
        load_full_feature_processed as shared_load_full_feature_processed,
        save_model_artifact as shared_save_model_artifact,
        summarize_fold_metrics as shared_summarize_fold_metrics,
    )

    TARGET_COL = shared_target_col
    if build_prediction_frame is None:
        build_prediction_frame = shared_build_prediction_frame
    if compute_regression_metrics is None:
        compute_regression_metrics = shared_compute_regression_metrics
    if ensure_family_result_dir is None:
        ensure_family_result_dir = shared_ensure_family_result_dir
    if load_full_feature_processed is None:
        load_full_feature_processed = shared_load_full_feature_processed
    if save_model_artifact is None:
        save_model_artifact = shared_save_model_artifact
    if summarize_fold_metrics is None:
        summarize_fold_metrics = shared_summarize_fold_metrics

    TabPFNRegressor = runtime_tabpfn_regressor
    ModelVersion = runtime_model_version
    return current_code_dir


In [ ]:
def expected_output_files() -> list[str]:
    return [
        "3_tabpfn_model_artifact_manifest.csv",
        "3_tabpfn_predictions_outer_test.csv",
        "3_tabpfn_metrics_by_model_by_outer_fold.csv",
        "3_tabpfn_metrics_by_model_mean_std.csv",
        "3_tabpfn_best_models.csv",
    ]


In [ ]:
def prepare_tabpfn_feature_frames(
    train_df: pd.DataFrame,
    other_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    x_train = train_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    x_other = other_df.loc[:, list(feature_cols)].apply(pd.to_numeric, errors="coerce")
    medians = x_train.median(axis=0).fillna(0.0)
    x_train = x_train.fillna(medians)
    x_other = x_other.fillna(medians)
    y_train = pd.Series(pd.to_numeric(train_df[TARGET_COL], errors="raise"), index=train_df.index).astype(float)
    return x_train, x_other, y_train


In [ ]:
def load_tabpfn_fold_bundle(step4_root: Path, outer_fold: int) -> dict[str, object]:
    if any(
        dep is None
        for dep in (
            build_prediction_frame,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
            TabPFNRegressor,
            ModelVersion,
        )
    ):
        initialize_runtime()

    train_df, test_df, inner_index, feature_cols = load_full_feature_processed(
        Path(step4_root),
        outer_fold=int(outer_fold),
    )
    return {
        "outer_fold": int(outer_fold),
        "train_df": train_df,
        "test_df": test_df,
        "inner_index": inner_index,
        "feature_cols": list(feature_cols),
    }


In [ ]:
def resolve_model_version():
    if hasattr(ModelVersion, "V2_6"):
        return getattr(ModelVersion, "V2_6")
    if hasattr(ModelVersion, "V2_5"):
        return ModelVersion.V2_5
    return ModelVersion.V2


In [ ]:
def predict_in_batches(model, features, batch_size: int = 4096, progress_every: int = 4) -> np.ndarray:
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)

    if batch_size is None or int(batch_size) <= 0 or int(batch_size) >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)

    predictions: list[np.ndarray] = []
    n_batches = (n_rows - 1) // int(batch_size) + 1
    for batch_idx in range(n_batches):
        start = batch_idx * int(batch_size)
        end = min((batch_idx + 1) * int(batch_size), n_rows)
        batch_features = features.iloc[start:end] if hasattr(features, "iloc") else features[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % int(progress_every) == 0) or (batch_idx + 1 == n_batches)):
            print(f"Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)")

    return np.concatenate(predictions, axis=0)


In [ ]:
def build_tabpfn_regressor(random_seed: int):
    model_version = resolve_model_version()
    device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    return TabPFNRegressor.create_default_for_version(
        model_version,
        n_estimators=TABPFN_N_ESTIMATORS,
        device=device,
        fit_mode="low_memory",
        memory_saving_mode="auto",
        ignore_pretraining_limits=True,
        inference_config={"SUBSAMPLE_SAMPLES": TABPFN_SUBSAMPLE_SAMPLES},
        random_state=int(random_seed),
    )


In [ ]:
def run_tabpfn_model_step(fold_bundle: dict[str, object], artifact_dir: Path) -> dict[str, object]:
    feature_cols = list(fold_bundle["feature_cols"])
    train_df = pd.DataFrame(fold_bundle["train_df"])
    test_df = pd.DataFrame(fold_bundle["test_df"])
    outer_fold = int(fold_bundle["outer_fold"])

    x_train, x_test, y_train = prepare_tabpfn_feature_frames(train_df, test_df, feature_cols)

    model = build_tabpfn_regressor(random_seed=SEED)
    model.fit(x_train, y_train.to_numpy(dtype=float))
    prediction_log = predict_in_batches(
        model,
        x_test,
        batch_size=BATCH_SIZE,
        progress_every=PREDICT_PROGRESS_EVERY,
    )
    prediction_frame = build_prediction_frame(
        test_df=test_df,
        prediction_log=prediction_log,
        outer_fold=outer_fold,
        model_family=MODEL_FAMILY,
        model_id=MODEL_ID,
        run_role=RUN_ROLE,
    )

    artifact_path = artifact_dir / f"fold_{outer_fold}" / f"{MODEL_ID}_{RUN_ROLE}.joblib"
    save_model_artifact(model, artifact_path)
    if not artifact_path.is_file():
        raise RuntimeError(f"artifact path does not exist after save: {artifact_path}")

    metric_row = {
        "outer_fold": outer_fold,
        "model_family": MODEL_FAMILY,
        "model_id": MODEL_ID,
        "run_role": RUN_ROLE,
        **compute_regression_metrics(prediction_frame),
    }
    artifact_row = {
        "outer_fold": outer_fold,
        "model_family": MODEL_FAMILY,
        "model_id": MODEL_ID,
        "run_role": RUN_ROLE,
        "artifact_path": str(artifact_path),
    }

    del model
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "prediction_frame": prediction_frame,
        "metric_row": metric_row,
        "artifact_row": artifact_row,
    }


In [ ]:
def write_tabpfn_family_outputs(
    result_dir: Path,
    prediction_frames: list[pd.DataFrame],
    metric_rows: list[dict[str, object]],
    artifact_rows: list[dict[str, object]],
) -> None:
    if not metric_rows or not prediction_frames or not artifact_rows:
        raise RuntimeError("TabPFN family compare produced no outputs")

    metrics_df = pd.DataFrame(metric_rows)
    summary_df = summarize_fold_metrics(metrics_df)
    best_models_df = summary_df.head(1).copy()

    pd.DataFrame(artifact_rows).to_csv(result_dir / "3_tabpfn_model_artifact_manifest.csv", index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(
        result_dir / "3_tabpfn_predictions_outer_test.csv",
        index=False,
    )
    metrics_df.to_csv(result_dir / "3_tabpfn_metrics_by_model_by_outer_fold.csv", index=False)
    summary_df.to_csv(result_dir / "3_tabpfn_metrics_by_model_mean_std.csv", index=False)
    best_models_df.to_csv(result_dir / "3_tabpfn_best_models.csv", index=False)


In [ ]:
def _assert_runtime_outputs_written(result_dir: Path) -> None:
    expected_paths = [result_dir / file_name for file_name in expected_output_files()]
    missing_paths = [str(path) for path in expected_paths if not path.is_file()]
    if missing_paths:
        raise RuntimeError(f"missing TabPFN runtime output files: {missing_paths}")

    for csv_path in expected_paths:
        frame = pd.read_csv(csv_path)
        if frame.empty:
            raise RuntimeError(f"runtime output must not be empty: {csv_path}")


In [ ]:
def run_tabpfn_family(step4_root: Path, step5_root: Path) -> None:
    if any(
        dep is None
        for dep in (
            build_prediction_frame,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
            TabPFNRegressor,
            ModelVersion,
        )
    ):
        initialize_runtime()

    result_dir = ensure_family_result_dir(Path(step5_root), "tabpfn")
    artifact_dir = result_dir / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, object]] = []
    artifact_rows: list[dict[str, object]] = []

    for outer_fold in OUTER_FOLDS:
        fold_bundle = load_tabpfn_fold_bundle(step4_root=step4_root, outer_fold=int(outer_fold))
        result = run_tabpfn_model_step(fold_bundle, artifact_dir)
        prediction_frames.append(result["prediction_frame"])
        metric_rows.append(result["metric_row"])
        artifact_rows.append(result["artifact_row"])

    write_tabpfn_family_outputs(result_dir, prediction_frames, metric_rows, artifact_rows)


In [ ]:
def run_fold(outer_fold: int) -> None:
    outer_fold = int(outer_fold)
    if outer_fold not in OUTER_FOLDS:
        raise RuntimeError(f"outer_fold must be one of {OUTER_FOLDS}, got {outer_fold}")

    fold_bundle = load_tabpfn_fold_bundle(step4_root=step4_root, outer_fold=outer_fold)
    fold_result = run_tabpfn_model_step(fold_bundle, tabpfn_artifact_dir)
    tabpfn_fold_results_by_fold[outer_fold] = fold_result
    print(
        {
            "step": "tabpfn_fold_completed",
            "outer_fold": outer_fold,
            "completed_folds": sorted(tabpfn_fold_results_by_fold),
        }
    )


In [ ]:
# --- Step 1: Resolve release-local Step4 inputs and Step5 outputs ---
current_code_dir = initialize_runtime()
step5_root = current_code_dir.parent
release_root = step5_root.parent
step4_root = release_root / "Step4_ModelInputs" / "2_Results"
tabpfn_result_dir = ensure_family_result_dir(step5_root, "tabpfn")
tabpfn_artifact_dir = tabpfn_result_dir / "artifacts"
tabpfn_artifact_dir.mkdir(parents=True, exist_ok=True)
tabpfn_fold_results_by_fold: dict[int, dict[str, object]] = {}
print({"step": "tabpfn_runtime_ready", "step4_root": str(step4_root), "result_dir": str(tabpfn_result_dir)})


In [ ]:
# --- Step 2: Define fold-wise runtime entrypoint ---
print({"step": "tabpfn_fold_runner_ready", "available_folds": list(OUTER_FOLDS), "batch_size": BATCH_SIZE})


In [ ]:
# --- Step 3: Run fold 1 ---
run_fold(1)


In [ ]:
# --- Step 4: Run fold 2 ---
run_fold(2)


In [ ]:
# --- Step 5: Run fold 3 ---
run_fold(3)


In [ ]:
# --- Step 6: Run fold 4 ---
run_fold(4)


In [ ]:
# --- Step 7: Run fold 5 ---
run_fold(5)


In [ ]:
# --- Step 8: Write and verify TabPFN outputs ---
completed_folds = sorted(int(fold) for fold in tabpfn_fold_results_by_fold)
missing_folds = [int(fold) for fold in OUTER_FOLDS if int(fold) not in tabpfn_fold_results_by_fold]
if missing_folds:
    raise RuntimeError(f"cannot finalize TabPFN outputs before all folds are completed: missing={missing_folds}")

tabpfn_prediction_frames = [
    tabpfn_fold_results_by_fold[int(outer_fold)]["prediction_frame"]
    for outer_fold in completed_folds
]
tabpfn_metric_rows = [
    tabpfn_fold_results_by_fold[int(outer_fold)]["metric_row"]
    for outer_fold in completed_folds
]
tabpfn_artifact_rows = [
    tabpfn_fold_results_by_fold[int(outer_fold)]["artifact_row"]
    for outer_fold in completed_folds
]

write_tabpfn_family_outputs(tabpfn_result_dir, tabpfn_prediction_frames, tabpfn_metric_rows, tabpfn_artifact_rows)
_assert_runtime_outputs_written(tabpfn_result_dir)
print({"step": "tabpfn_outputs_written", "files": expected_output_files(), "metric_rows": len(tabpfn_metric_rows), "completed_folds": completed_folds})
